# Fine-Tuning Large Language Models with OpenAI API

This notebook sets up and sends a request to the OpenAI API to fine-tune a model. The process involves configuring the API endpoint, loading environment variables for authentication, and defining the data payload for the fine-tuning job. 

In [2]:
import requests
from openai import OpenAI
import os
from dotenv import load_dotenv
import json
import pandas as pd
from datetime import datetime


# Load OPENAI_API_KEY from .env file
load_dotenv()

# Initialize the OpenAI client
client = OpenAI()

## Transform data from local format to jsonl

In [3]:
PROMPT_TEMPLATE = "Do the two product descriptions refer to the same real-world product? Entity 1: 'Entity 1'. Entity 2: 'Entity 2'."
MODEL_NAME = "gpt-4.1-mini-2025-04-14"
VALIDATION_FILE_ID = "file-TTmjCB7a6yLzPyujC4xZa5"

In [4]:
def insert_product_descriptions(prompt_template: str, product1: str, product2: str):
    # Replace placeholder texts with actual product descriptions
    prompt = prompt_template.replace("'Entity 1'", product1).replace("'Entity 2'", product2)
    return prompt

def create_training_example(prompt_template: str, product1: str, product2: str, label: int):
    # Create the prompt with product descriptions
    prompt = insert_product_descriptions(prompt_template, product1, product2)
    if label == 1 or label == "1":
        label = "Yes"
    elif label == 0 or label == "0":
        label = "No"
    else:
        ValueError("Label is not 0 or 1")
    
    # Create the training example in the format required for fine-tuning
    return {
        "messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": label}
        ]
    }


In [5]:
def train_based_on_csv(train_set_path: str, run_name: str, model_name: str, validation_file_id: str):
    # Load the test set
    train_set = pd.read_csv(train_set_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"../../results/{model_name}/{run_name}/{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    # Create training examples
    training_examples = []
    print(f"Creating training examples for {run_name}. The dataset has {len(train_set)} pairs each.")

    
    for index, row in train_set.iterrows():
        prompt = row["prompt"]
        label = row["completion"]
        
        example = {
            "messages": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": label}
            ]
        }
        
        training_examples.append(example)

    # Save the training file
    training_file_path = os.path.join(output_dir, "training.jsonl")
    with open(training_file_path, "w") as f:
        for example in training_examples:
            f.write(json.dumps(example) + "\n")
        
    # Upload the training file using the SDK
    training_file = client.files.create(
        file=open(training_file_path, "rb"),
        purpose="fine-tune"
    )

    print(f"Training file uploaded successfully. File ID: {training_file.id}")

    # Start the fine-tuning job using the SDK
    fine_tune_job = client.fine_tuning.jobs.create(
        training_file=training_file.id,
        validation_file=validation_file_id,
        model=model_name,
        hyperparameters={
            "n_epochs": 5
        },
        seed=42
    )

    print(f"Fine-tuning job started successfully. Job ID: {fine_tune_job.id}")

    # Save the run configuration
    run_config = {
        "run_name": run_name,
        "timestamp": timestamp,
        "model": MODEL_NAME,
        "file_id": training_file.id,
        "job_id": fine_tune_job.id,
        "status": fine_tune_job.status,
        "created_at": fine_tune_job.created_at
    }

    with open(os.path.join(output_dir, "fine-tune-run_config.json"), "w") as f:
        json.dump(run_config, f, indent=2)

    print(f"Run configuration and training files saved to: {output_dir}")

In [6]:
def train_based_on_pickle(train_set_path: str, run_name: str, model_name: str, validation_file_id: str):
    # Load the test set
    train_set = pd.read_pickle(train_set_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"../../results/{model_name}/{run_name}/{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    # Create training examples
    training_examples = []
    print(f"Creating training examples for {run_name}. The dataset has {len(train_set)} pairs each.")

        
    for index, row in train_set.iterrows():
        product1, product2 = row['title_left'], row['title_right']
        label = str(row.get('label'))  # Convert label to string
        
        example = create_training_example(PROMPT_TEMPLATE, product1, product2, label)
        training_examples.append(example)

    # Save the training file
    training_file_path = os.path.join(output_dir, f"training-{run_name}.jsonl")
    with open(training_file_path, "w") as f:
        for example in training_examples:
            f.write(json.dumps(example) + "\n")


    # Upload the training file using the SDK
    training_file = client.files.create(
        file=open(training_file_path, "rb"),
        purpose="fine-tune"
    )

    print(f"Training file uploaded successfully. File ID: {training_file.id}")

    # Start the fine-tuning job using the SDK
    fine_tune_job = client.fine_tuning.jobs.create(
        training_file=training_file.id,
        validation_file=validation_file_id,
        model=model_name,
        hyperparameters={
            "n_epochs": 5
        },
        seed=42
    )

    print(f"Fine-tuning job started successfully. Job ID: {fine_tune_job.id}")

    # Save the run configuration
    run_config = {
        "run_name": run_name,
        "timestamp": timestamp,
        "model": MODEL_NAME,
        "file_id": training_file.id,
        "job_id": fine_tune_job.id,
        "status": fine_tune_job.status,
        "created_at": fine_tune_job.created_at
    }

    with open(os.path.join(output_dir, "fine-tune-run_config.json"), "w") as f:
        json.dump(run_config, f, indent=2)

    print(f"Run configuration and training files saved to: {output_dir}")

## Upload validation file

In [12]:
validation_set_path = "../../data/wdc/validation/preprocessed_wdcproducts80cc20rnd000un_valid_small.pkl.gz"
# Load the test set
validation_set = pd.read_pickle(validation_set_path)

# Create training examples
validation_examples = []
print(f"The validation set has {len(validation_set)} pairs each.")

for index, row in validation_set.iterrows():
    product1, product2 = row['title_left'], row['title_right']
    label = str(row.get('label'))  # Convert label to string
    pair_id = row['pair_id']
    
    example = create_training_example(PROMPT_TEMPLATE, product1, product2, label)
    validation_examples.append(example)

# Save the training file
with open(validation_set_path.replace(".pkl.gz", ".jsonl"), "w") as f:
    for example in validation_examples:
        f.write(json.dumps(example) + "\n")


# Upload the training file using the SDK
validation_file = client.files.create(
    file=open(validation_set_path.replace(".pkl.gz", ".jsonl"), "rb"),
    purpose="fine-tune"
)

print(f"Validation file uploaded successfully. File ID: {validation_file.id}")

The validation set has 2500 pairs each.
Validation file uploaded successfully. File ID: file-TTmjCB7a6yLzPyujC4xZa5


## Small wdc no augmentations

In [ ]:
# Load the test set
train_set = pd.read_pickle("../../data/wdc/train_small/preprocessed_wdcproducts80cc20rnd000un_train_small.pkl.gz")

# Create output directory structure
run_name = "fine-tune-wdc-small-regular"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../../results/{MODEL_NAME}/{run_name}/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Create training examples
training_examples = []
print(f"Creating training examples for {run_name}. The dataset has {len(train_set)} pairs each.")

    
for index, row in train_set.iterrows():
    product1, product2 = row['title_left'], row['title_right']
    label = str(row.get('label'))  # Convert label to string
    pair_id = row['pair_id']
    
    example = create_training_example(PROMPT_TEMPLATE, product1, product2, label)
    training_examples.append(example)

# Save the training file
training_file_path = os.path.join(output_dir, f"training-{run_name}.jsonl")
with open(training_file_path, "w") as f:
    for example in training_examples:
        f.write(json.dumps(example) + "\n")


# Upload the training file using the SDK
training_file = client.files.create(
    file=open(training_file_path, "rb"),
    purpose="fine-tune"
)

print(f"Training file uploaded successfully. File ID: {training_file.id}")

# Start the fine-tuning job using the SDK
fine_tune_job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    model=MODEL_NAME,
    hyperparameters={
        "n_epochs": 5
    },
    seed=42
)

print(f"Fine-tuning job started successfully. Job ID: {fine_tune_job.id}")

# Save the run configuration
run_config = {
    "run_name": run_name,
    "timestamp": timestamp,
    "model": MODEL_NAME,
    "file_id": training_file.id,
    "job_id": fine_tune_job.id,
    "status": fine_tune_job.status,
    "created_at": fine_tune_job.created_at
}

with open(os.path.join(output_dir, "fine-tune-run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Run configuration and training files saved to: {output_dir}")

Creating training examples for fine-tune-wdc-small-regular. The dataset has 2500 pairs each.
Training file uploaded successfully. File ID: file-JeW9ZhtXCEdLXWGu3LYitg
Fine-tuning job started successfully. Job ID: ftjob-rILHCoiqEgEDzWiySiu3e15K
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809


## Train simple swapping

In [17]:
train_based_on_csv("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_swapped_matching_examples.csv", "fine-tune-wdc-simple-swapping", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for fine-tune-wdc-simple-swapping. The dataset has 4137 pairs each.
Training file uploaded successfully. File ID: file-4LmWTpKouBd7ETXZQV6ELT
Fine-tuning job started successfully. Job ID: ftjob-EuFcxHRbchLk42TePoaD76FT
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-simple-swapping/20250508_204823


## Train based on 25% swapping

In [4]:
# Load the test set
train_set = pd.read_csv("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_explanations_40_swapped_matching_examples_0_25_permutations_v2.csv")

# Create output directory structure
run_name = "fine-tune-wdc-swapping_0_25"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../../results/{MODEL_NAME}/{run_name}/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Create training examples
training_examples = []
print(f"Creating training examples for {run_name}. The dataset has {len(train_set)} pairs each.")

    
for index, row in train_set.iterrows():
    prompt = row["prompt"]
    label = row["completion"]
    pair_id = row['id']
    
    example = {
        "messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": label}
        ]
    }
    
    training_examples.append(example)

# Save the training file
training_file_path = os.path.join(output_dir, "training.jsonl")
with open(training_file_path, "w") as f:
    for example in training_examples:
        f.write(json.dumps(example) + "\n")
        
# Upload the training file using the SDK
training_file = client.files.create(
    file=open(training_file_path, "rb"),
    purpose="fine-tune"
)

print(f"Training file uploaded successfully. File ID: {training_file.id}")

# Start the fine-tuning job using the SDK
fine_tune_job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    model=MODEL_NAME,
    hyperparameters={
        "n_epochs": 5
    },
    seed=42
)

print(f"Fine-tuning job started successfully. Job ID: {fine_tune_job.id}")

# Save the run configuration
run_config = {
    "run_name": run_name,
    "timestamp": timestamp,
    "model": MODEL_NAME,
    "file_id": training_file.id,
    "job_id": fine_tune_job.id,
    "status": fine_tune_job.status,
    "created_at": fine_tune_job.created_at
}

with open(os.path.join(output_dir, "fine-tune-run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Run configuration and training files saved to: {output_dir}")

Creating training examples for fine-tune-wdc-swapping_0_25. The dataset has 5234 pairs each.
Training file uploaded successfully. File ID: file-P7FXzT1rekpmCQpjg61tsM
Fine-tuning job started successfully. Job ID: ftjob-S4lsDkw3c0lEMiUy5tR0J0yn
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-swapping_0_25/20250507_155922


## 100% swapping

In [7]:
train_based_on_csv("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_explanations_40_swapped_matching_examples_all_attributes.csv", "fine-tune-wdc-100-swapping", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for fine-tune-wdc-100-swapping. The dataset has 4137 pairs each.
Training file uploaded successfully. File ID: file-7a3c8uh6VqJeRpmdkqgLD4
Fine-tuning job started successfully. Job ID: ftjob-ypcJXLMA4N8dwrSa1sbPeCTG
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-100-swapping/20250509_103256


## Nplaug augmentation

In [14]:
train_based_on_csv("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_all_augmentation_nplaug_matching_examples_v2.csv", "fine-tune-wdc-nplaug", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for fine-tune-wdc-nplaug. The dataset has 3920 pairs each.
Training file uploaded successfully. File ID: file-46DXhbe255kjr2inNEEQNn
Fine-tuning job started successfully. Job ID: ftjob-JclCHln60me742x0E08fNb2b
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-nplaug/20250508_204534


## Train based on random swapping augmentation 

In [19]:
train_based_on_pickle("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_random_word_swap_1_1.pkl.gz", "fine-tune-wdc-random-swapping", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for fine-tune-wdc-random-swapping. The dataset has 4000 pairs each.
Training file uploaded successfully. File ID: file-UAnF2oBD83msnBnBuw4pm3
Fine-tuning job started successfully. Job ID: ftjob-oDiAc97YfNN4r6Syt9Tbyi49
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-random-swapping/20250508_212306


## Train basic upsample 

In [6]:
train_based_on_csv("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_upsampled.csv", "fine-tune-wdc-basic-upsample", MODEL_NAME, VALIDATION_FILE_ID)

Creating training examples for fine-tune-wdc-basic-upsample. The dataset has 4000 pairs each.
Training file uploaded successfully. File ID: file-FEA4XwVB8q7DUbt5V2gngN
Fine-tuning job started successfully. Job ID: ftjob-waYhWf5alFi9oibPNAFJxTzD
Run configuration and training files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-basic-upsample/20250508_231510
